In [1]:
import keras
import numpy as np
import pandas as pd

from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

print("All libraries imported successfully!")

I0000 00:00:1795154830.746369    3457 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1795154830.778172    3457 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1795154832.555340    3457 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1795154837.215600    3457 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

All libraries imported successfully!


Build an ANN: Create an AI model that learns patterns in data to predict wine quality. Tune hyperparameters: Adjust settings like learning rate to improve its predictions.

2. Track experiments using MLflow: Record each model's settings and results so we can compare them and choose the best-performing model.

We're loading wine data containing chemical measurements and quality scores. Our ANN will learn to predict wine quality from these measurements.

In [2]:
data = pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";"
)

print("Dataset shape:", data.shape)
data.head()

Dataset shape: (4898, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


Your dataset contains 4,898 wine samples and 12 columns: 11 chemical measurements (features) and 1 wine quality score (target).

 We'll train an Artificial Neural Network (ANN) to learn how these chemical measurements relate to wine quality and predict the quality of new wines.

In [3]:
display(data.tail())
display(data.describe())

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7
4897,6.0,0.21,0.38,0.8,0.020,22.0,98.0,0.98941,3.26,0.32,11.8,6


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
count,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000,4898.000000
mean,6.854788,0.278241,0.334192,6.391415,0.045772,35.308085,138.360657,0.994027,3.188267,0.489847,10.514267,5.877909
std,0.843868,0.100795,0.121020,5.072058,0.021848,17.007137,42.498065,0.002991,0.151001,0.114126,1.230621,0.885639
min,3.800000,0.080000,0.000000,0.600000,0.009000,2.000000,9.000000,0.987110,2.720000,0.220000,8.000000,3.000000
25%,6.300000,0.210000,0.270000,1.700000,0.036000,23.000000,108.000000,0.991723,3.090000,0.410000,9.500000,5.000000
50%,6.800000,0.260000,0.320000,5.200000,0.043000,34.000000,134.000000,0.993740,3.180000,0.470000,10.400000,6.000000
75%,7.300000,0.320000,0.390000,9.900000,0.050000,46.000000,167.000000,0.996100,3.280000,0.550000,11.400000,6.000000
max,14.200000,1.100000,1.660000,65.800000,0.346000,289.000000,440.000000,1.038980,3.820000,1.080000,14.200000,9.000000


This helps us understand the range, average and variation of the chemical measurements before training our ANN.

split the dataset
Now we'll divide the wine dataset into training (75%) and testing (25%) data.

In [4]:
train, test = train_test_split(
    data,
    test_size=0.25,
    random_state=42
)

print("Training samples:", train.shape)
print("Testing samples:", test.shape)

Training samples: (3673, 12)
Testing samples: (1225, 12)


The model learns from the training data, while the test data is kept separate to evaluate its performance on unseen examples.

Separate Features and Target 
Now we'll separate the chemical measurements (X) from the wine quality score (y).

In [5]:
train_x = train.drop(["quality"], axis=1).values
train_y = train[["quality"]].values.ravel()

test_x = test.drop(["quality"], axis=1).values
test_y = test[["quality"]].values.ravel()

print("Training features:", train_x.shape)
print("Training targets:", train_y.shape)
print("Testing features:", test_x.shape)
print("Testing targets:", test_y.shape)

Training features: (3673, 11)
Training targets: (3673,)
Testing features: (1225, 11)
Testing targets: (1225,)


Create the Validation Dataset
Now we'll split the training data again, keeping 20% for validation

In [6]:
train_x, valid_x, train_y, valid_y = train_test_split(
    train_x,
    train_y,
    test_size=0.20,
    random_state=42
)

print("Training:", train_x.shape)
print("Validation:", valid_x.shape)
print("Testing:", test_x.shape)

Training: (2938, 11)
Validation: (735, 11)
Testing: (1225, 11)


Training: Teaches the ANN to predict wine quality.
Validation: Helps us tune the model and compare settings.
Testing: Checks the final model on unseen data.

In [7]:
signature = infer_signature(train_x, train_y)

print("Number of input features:", train_x.shape[1])
print("Average of each feature:", np.mean(train_x, axis=0))

Number of input features: 11
Average of each feature: [6.86621852e+00 2.80377808e-01 3.32597005e-01 6.42164738e+00
 4.55513955e-02 3.53556841e+01 1.38792376e+02 9.94074221e-01
 3.18919333e+00 4.88396869e-01 1.05005673e+01]


The model signature describes the expected inputs and outputs, helping MLflow save and manage the model correctly.

Build Your Artificial Neural Network (ANN)

In [8]:
def build_model(params):
    mean = np.mean(train_x, axis=0)
    var = np.var(train_x, axis=0)

    model = keras.Sequential([
        keras.Input(shape=(train_x.shape[1],)),
        keras.layers.Normalization(mean=mean, variance=var),
        keras.layers.Dense(64, activation="relu"),
        keras.layers.Dense(1)
    ])

    model.compile(
        optimizer=keras.optimizers.SGD(
            learning_rate=params["lr"],
            momentum=params["momentum"]
        ),
        loss="mean_squared_error",
        metrics=[keras.metrics.RootMeanSquaredError()]
    )

    return model

print("ANN architecture defined successfully!")

ANN architecture defined successfully!


We're defining a neural network with 11 inputs, 64 neurons in a hidden layer and one output that predicts wine quality. The learning rate and momentum will be tuned later.

Now we'll train the model and record its performance.

In [9]:
params = {
    "lr": 0.01,
    "momentum": 0.5
}

mlflow.set_experiment("wine-quality")

with mlflow.start_run():
    model = build_model(params)

    model.fit(
        train_x,
        train_y,
        validation_data=(valid_x, valid_y),
        epochs=3,
        batch_size=64
    )

    results = model.evaluate(valid_x, valid_y)

    mlflow.log_params(params)
    mlflow.log_metric("eval_rmse", results[1])

    print("Validation RMSE:", results[1])

2026/11/20 06:16:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/20 06:16:34 INFO mlflow.store.db.utils: Updating database tables
2026/11/20 06:16:35 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


Epoch 1/3


E0000 00:00:1795155396.593940    3457 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


46/46 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 4.2535 - root_mean_squared_error: 2.0624 - val_loss: 1.2840 - val_root_mean_squared_error: 1.1331
Epoch 2/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1.0402 - root_mean_squared_error: 1.0199 - val_loss: 0.9510 - val_root_mean_squared_error: 0.9752
Epoch 3/3
46/46 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.8116 - root_mean_squared_error: 0.9009 - val_loss: 0.7757 - val_root_mean_squared_error: 0.8807
23/23 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.7757 - root_mean_squared_error: 0.8807 
Validation RMSE: 0.8807340264320374


We're training the ANN for three epochs and recording its learning rate, momentum and validation error in MLflow.

RMSE measures prediction error; lower is better.

our validation RMSE decreased from 1.1331 to 0.8807 over three epochs. Lower RMSE means better predictions. The CUDA warning is not a problem here because training completed on the CPU.

Define Hyperparameter Search
Now we'll let Hyperopt explore different learning rates and momentum values to find better settings.

In [10]:
space = {
    "lr": hp.loguniform("lr", np.log(1e-5), np.log(1e-1)),
    "momentum": hp.uniform("momentum", 0.0, 1.0)
}

print("Hyperparameter search space defined!")

Hyperparameter search space defined!


Instead of manually choosing settings, Hyperopt will search for combinations that reduce the model's prediction error.

Connect Hyperparameter Tuning to MLflow
Now we'll create a function that trains the ANN using different hyperparameters and records each experiment in MLflow.

In [11]:
def objective(params):
    with mlflow.start_run(nested=True):
        model = build_model(params)

        model.fit(
            train_x,
            train_y,
            validation_data=(valid_x, valid_y),
            epochs=3,
            batch_size=64,
            verbose=0
        )

        results = model.evaluate(valid_x, valid_y, verbose=0)
        rmse = float(results[1])

        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse", rmse)

        return {
            "loss": rmse,
            "status": STATUS_OK,
            "model": model
        }

print("MLflow tuning function created!")

MLflow tuning function created!


Every time Hyperopt tries a new learning rate and momentum, this function will train the ANN and record its validation RMSE in MLflow.

Run Hyperparameter Tuning
Now we'll train 4 ANN models with different learning rates and momentum values to find the best combination.

In [12]:
mlflow.set_experiment("wine-quality")

with mlflow.start_run(run_name="hyperparameter-tuning"):
    trials = Trials()

    best = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=4,
        trials=trials
    )

    best_run = min(trials.results, key=lambda x: x["loss"])

    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])

    print("Best parameters:", best)
    print("Best validation RMSE:", best_run["loss"])

100%|██████████| 4/4 [00:03<00:00,  1.03trial/s, best loss: 1.5269891023635864]
Best parameters: {'lr': np.float64(0.00041721528377744563), 'momentum': np.float64(0.7068593528933922)}
Best validation RMSE: 1.5269891023635864


Hyperopt will test four combinations, and MLflow will record their parameters and prediction errors so we can compare them.

Metric	Result
Best learning rate	0.000417
Best momentum	0.7069
Best tuned validation RMSE	1.527
Previous ANN validation RMSE	0.881

Important: Your original ANN performed better than all four tuned models because its RMSE was lower. This can happen when only a few hyperparameter combinations are tested.

In [13]:
print("MLflow Tracking URI:", mlflow.get_tracking_uri())

experiment = mlflow.get_experiment_by_name("wine-quality")

print("Experiment:", experiment)

MLflow Tracking URI: sqlite:////workspaces/MLIPr/week-07/mlflow.db
Experiment: <Experiment: artifact_location='/workspaces/MLIPr/week-07/mlruns/1', creation_time=1795155395940, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1795155395940, lifecycle_stage='active', name='wine-quality', tags={}, trace_location=None, workspace='default'>


In [14]:
with mlflow.start_run(run_name="best-original-ann"):
    mlflow.log_param("lr", 0.01)
    mlflow.log_param("momentum", 0.5)
    mlflow.log_metric("eval_rmse", float(results[1]))

    mlflow.tensorflow.log_model(
        model,
        name="model",
        signature=signature
    )

print("Best ANN saved in MLflow!")

Best ANN saved in MLflow!


In [15]:
runs = mlflow.search_runs(
    experiment_names=["wine-quality"],
    filter_string="tags.mlflow.runName = 'best-original-ann'"
)

best_run_id = runs.iloc[0]["run_id"]

registered_model = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="WineQualityANN"
)

print("Registered model:", registered_model.name)
print("Version:", registered_model.version)

Successfully registered model 'WineQualityANN'.
2026/11/20 06:39:31 WARNING mlflow.tracking._model_registry.fluent: Run with id 1a5174f019b94b2e9220fc755cef8619 has no artifacts at artifact path 'model', registering model based on models:/m-91600313d1f447e3bc6b9f1aa494854e instead


Registered model: WineQualityANN
Version: 1


Created version '1' of model 'WineQualityANN'.


Model registration allows you to save and manage versions of your trained ANN for future use.